In [5]:
# %% CELL 18: PROPERLY SCALED MLP (The "Gold Standard" Fix)
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import os

print("🧠 Launching PROPERLY SCALED MLP (Z-Score Standardization)...")

# 1. SETUP
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.backends.mps.is_available(): DEVICE = torch.device("mps")
print(f"   Using Device: {DEVICE}")

# 2. LOAD DATA
LOAD_PATH = '../data/processed'
try:
    if os.path.exists(os.path.join(LOAD_PATH, 'X_tr_scaled.npy')):
        X_train = np.load(os.path.join(LOAD_PATH, 'X_tr_scaled.npy'))
        X_val   = np.load(os.path.join(LOAD_PATH, 'X_val_scaled.npy'))
        X_test  = np.load(os.path.join(LOAD_PATH, 'X_test_scaled.npy'))
    else:
        # Fallback to surgical removal
        print("   ⚠️ Clean data not found. Loading 'X_tr_final.npy' and stripping clusters...")
        X_train = np.load(os.path.join(LOAD_PATH, 'X_tr_final.npy'))[:, :-7]
        X_val   = np.load(os.path.join(LOAD_PATH, 'X_val_final.npy'))[:, :-7]
        X_test  = np.load(os.path.join(LOAD_PATH, 'X_test_final.npy'))[:, :-7]
        
    y_tr    = np.load(os.path.join(LOAD_PATH, 'y_tr.npy'))
    y_val   = np.load(os.path.join(LOAD_PATH, 'y_val.npy'))
    test_ids = np.load(os.path.join(LOAD_PATH, 'test_ids.npy'), allow_pickle=True)
    test_df = pd.DataFrame({'id': test_ids})
    
    print(f"   ✅ Data Loaded. X_train shape: {X_train.shape}")

except Exception as e:
    raise FileNotFoundError(f"❌ Data load failed: {e}")

# --- FIX: STANDARDIZE TARGETS (Z-Score) ---
# Compute statistics on TRAINING DATA ONLY (No Leakage)
target_mean = y_tr.mean(axis=0, keepdims=True)  # Shape: (1, 3)
target_std  = y_tr.std(axis=0, keepdims=True)   # Shape: (1, 3)

print(f"   📊 Target Statistics (computed on train only):")
print(f"      Mean: {target_mean[0]}")
print(f"      Std:  {target_std[0]}")

# Standardize: (x - mean) / std
# This makes targets have Mean=0, Std=1 (Range approx -3 to +3)
y_tr_standardized  = (y_tr - target_mean) / target_std
y_val_standardized = (y_val - target_mean) / target_std 

# Convert to tensors
X_train_tensor = torch.FloatTensor(np.nan_to_num(X_train, nan=0.0))
y_train_tensor = torch.FloatTensor(y_tr_standardized)
X_val_tensor   = torch.FloatTensor(np.nan_to_num(X_val, nan=0.0))
y_val_tensor   = torch.FloatTensor(y_val_standardized)
X_test_tensor  = torch.FloatTensor(np.nan_to_num(X_test, nan=0.0))

class SimpleDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = X
        self.y = y
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        return (self.X[idx], self.y[idx]) if self.y is not None else self.X[idx]

BATCH_SIZE = 512
train_loader = DataLoader(SimpleDataset(X_train_tensor, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(SimpleDataset(X_val_tensor, y_val_tensor), batch_size=BATCH_SIZE*4, shuffle=False)
test_loader  = DataLoader(SimpleDataset(X_test_tensor), batch_size=BATCH_SIZE*4, shuffle=False)

# 3. ARCHITECTURE (Optimized)
class ImprovedMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.08),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            
            nn.Linear(64, 3) # Output
        )
    
    def forward(self, x):
        return self.net(x)

# 4. TRAINING
SEEDS = [42, 2024, 777]
mlp_preds = []

# MSE Loss is perfect for Z-Scored targets (Loss will be ~1.0)
criterion = nn.MSELoss()  

for seed in SEEDS:
    print(f"\n🌱 Seed: {seed}")
    torch.manual_seed(seed)
    
    model = ImprovedMLP(input_dim=X_train.shape[1]).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    
    best_score = float('inf')
    best_weights = None
    patience_counter = 0
    
    for epoch in range(80):
        model.train()
        train_loss = 0
        for X_b, y_b in train_loader:
            optimizer.zero_grad()
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            loss = criterion(model(X_b), y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        
        # Validation (DE-STANDARDIZE for scoring)
        model.eval()
        val_p, val_t = [], []
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
                pred_standardized = model(X_b).cpu().numpy()
                true_standardized = y_b.cpu().numpy()
                
                # REVERSE THE TRANSFORM: y = (z * std) + mean
                pred_original = (pred_standardized * target_std) + target_mean
                true_original = (true_standardized * target_std) + target_mean
                
                val_p.append(pred_original)
                val_t.append(true_original)
        
        vp, vt = np.vstack(val_p), np.vstack(val_t)
        mae = np.mean(np.abs(vp - vt), axis=0)
        score = (0.5 * mae[0]) + (0.3 * mae[1]) + (0.2 * mae[2])
        
        scheduler.step(score)
        
        if score < best_score:
            best_score = score
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
        
        if (epoch+1) % 10 == 0:
            # Train Loss should be around ~1.0 now
            print(f"   Epoch {epoch+1}/80 | Train Loss: {train_loss/len(train_loader):.4f} | Val Score: {score:.5f}")
        
        if patience_counter >= 12:
            print(f"   Early stopping at epoch {epoch+1}")
            break
    
    print(f"   🏆 Best Val Score: {best_score:.5f}")
    
    # Predict (and DE-STANDARDIZE)
    model.load_state_dict(best_weights)
    model.eval()
    seed_p = []
    with torch.no_grad():
        for X_b in test_loader:
            X_b = X_b.to(DEVICE)
            pred_standardized = model(X_b).cpu().numpy()
            # REVERSE TRANSFORM
            pred_original = (pred_standardized * target_std) + target_mean
            seed_p.append(pred_original)
    mlp_preds.append(np.vstack(seed_p))

# 5. SAVE
avg_mlp_preds = np.mean(mlp_preds, axis=0)
sub_mlp = pd.DataFrame({'id': test_df['id']})
sub_mlp[['target_short', 'target_medium', 'target_long']] = avg_mlp_preds

# Variance Check
std_check = sub_mlp[['target_short', 'target_medium', 'target_long']].values.std()
print(f"\n📊 Prediction Variance: {std_check:.6f} (Target > 0.001)")

# Center
for c in ['target_short', 'target_medium', 'target_long']:
    sub_mlp[c] -= sub_mlp[c].mean()

sub_mlp.to_csv('submission_mlp_improved.csv', index=False)
print("✅ Proper Scale MLP Complete. Saved 'submission_mlp_improved.csv'")

🧠 Launching PROPERLY SCALED MLP (Z-Score Standardization)...
   Using Device: mps
   ✅ Data Loaded. X_train shape: (125452, 214)
   📊 Target Statistics (computed on train only):
      Mean: [-2.9529589e-05 -1.6704464e-04 -6.6697545e-04]
      Std:  [0.00599921 0.01476223 0.0296532 ]

🌱 Seed: 42
   Epoch 10/80 | Train Loss: 1.0013 | Val Score: 0.00681
   Early stopping at epoch 16
   🏆 Best Val Score: 0.00679

🌱 Seed: 2024
   Epoch 10/80 | Train Loss: 1.0004 | Val Score: 0.00680
   Early stopping at epoch 15
   🏆 Best Val Score: 0.00677

🌱 Seed: 777
   Epoch 10/80 | Train Loss: 1.0008 | Val Score: 0.00684
   Epoch 20/80 | Train Loss: 1.0002 | Val Score: 0.00680
   Early stopping at epoch 23
   🏆 Best Val Score: 0.00679

📊 Prediction Variance: 0.000892 (Target > 0.001)
✅ Proper Scale MLP Complete. Saved 'submission_mlp_improved.csv'
